# Tarot Card Interaction Dynamics Generator

Generates interaction dynamics for all **78 x 77 = 6,006** directional card pairs across 5 dynamics:
- **Engendrar** (Generate) - how one energy creates the other
- **Conflito** (Conflict) - where energies collide
- **Estagnar** (Stagnate) - when movement ceases
- **Regressar** (Regress) - the path of return
- **Necessitar** (Need) - what one needs from the other

Uses **Gemini 2.5 Flash-Lite** (free tier: 15 RPM, 1,000 RPD) with the new `google-genai` SDK.

All output is saved to **Google Drive** (`My Drive/tarot-generator/`) so it persists across sessions.

### Setup
1. Get a **free** API key at https://aistudio.google.com/apikey
2. In Colab: click the key icon (left sidebar) -> Add secret `GEMINI_API_KEY`
3. Run all cells. Generation takes ~25-30 min with 20 pairs/batch.
4. If it stops (timeout, rate limit), just re-run - it resumes from the Drive checkpoint.
5. Result is saved at `My Drive/tarot-generator/tarot_data_full.json`

In [ ]:
# Install the new Google Gen AI SDK (the old google-generativeai is deprecated)
!pip install -q google-genai

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
import json
import time
import re
import os

# ============================================================
# CONFIGURATION
# ============================================================

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon on the left sidebar -> Add "GEMINI_API_KEY"
# Option 2: Paste your key directly below (less secure)
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = ""  # <-- Paste your key here if not using Colab Secrets

assert API_KEY, "Set your Gemini API key (free at https://aistudio.google.com/apikey)"

client = genai.Client(api_key=API_KEY)

# Model selection:
# - "gemini-2.5-flash-lite" : FREE tier, 15 RPM, 1000 RPD (recommended)
# - "gemini-2.5-flash"      : FREE tier, 10 RPM, 250 RPD (higher quality, slower)
# - "gemini-2.5-pro"        : FREE tier, 5 RPM, 100 RPD (best quality, very slow)
# If you have a paid tier, use "gemini-2.5-flash" for best quality/speed balance.
MODEL_NAME = "gemini-2.5-flash-lite"

# Batch size: more pairs per call = fewer API calls but longer responses.
# 20 pairs fits well within output token limits and keeps RPD under 1,000.
PAIRS_PER_BATCH = 20

# Delay between API calls (seconds). Adjust based on your tier:
# Free flash-lite (15 RPM) -> 4.5s | Free flash (10 RPM) -> 6.5s
DELAY_BETWEEN_CALLS = 4.5

# All files saved to Google Drive (persist across Colab sessions)
CHECKPOINT_FILE = os.path.join(DRIVE_FOLDER, "checkpoint.json")
OUTPUT_FILE = os.path.join(DRIVE_FOLDER, "tarot_data_full.json")

total_pairs = 78 * 77  # 6,006
total_batches = (total_pairs + PAIRS_PER_BATCH - 1) // PAIRS_PER_BATCH
est_minutes = total_batches * DELAY_BETWEEN_CALLS / 60

print(f"Model: {MODEL_NAME}")
print(f"Pairs per batch: {PAIRS_PER_BATCH}")
print(f"Total batches: {total_batches}")
print(f"Estimated time: ~{est_minutes:.0f} min ({est_minutes/60:.1f} h)")
print(f"Delay: {DELAY_BETWEEN_CALLS}s between calls")
print(f"Checkpoint: {CHECKPOINT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print("API key configured OK")

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
import json
import time
import re
import os

# ============================================================
# CONFIGURATION
# ============================================================

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon on the left sidebar -> Add "GEMINI_API_KEY"
# Option 2: Paste your key directly below (less secure)
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = ""  # <-- Paste your key here if not using Colab Secrets

assert API_KEY, "Set your Gemini API key (free at https://aistudio.google.com/apikey)"

client = genai.Client(api_key=API_KEY)

# Model selection:
# - "gemini-2.5-flash-lite" : FREE tier, 15 RPM, 1000 RPD (recommended)
# - "gemini-2.5-flash"      : FREE tier, 10 RPM, 250 RPD (higher quality, slower)
# - "gemini-2.5-pro"        : FREE tier, 5 RPM, 100 RPD (best quality, very slow)
# If you have a paid tier, use "gemini-2.5-flash" for best quality/speed balance.
MODEL_NAME = "gemini-2.5-flash-lite"

# Batch size: more pairs per call = fewer API calls but longer responses.
# 20 pairs fits well within output token limits and keeps RPD under 1,000.
PAIRS_PER_BATCH = 20

# Delay between API calls (seconds). Adjust based on your tier:
# Free flash-lite (15 RPM) -> 4.5s | Free flash (10 RPM) -> 6.5s
DELAY_BETWEEN_CALLS = 4.5

CHECKPOINT_FILE = "checkpoint.json"
OUTPUT_FILE = "tarot_data_full.json"

total_pairs = 78 * 77  # 6,006
total_batches = (total_pairs + PAIRS_PER_BATCH - 1) // PAIRS_PER_BATCH
est_minutes = total_batches * DELAY_BETWEEN_CALLS / 60

print(f"Model: {MODEL_NAME}")
print(f"Pairs per batch: {PAIRS_PER_BATCH}")
print(f"Total batches: {total_batches}")
print(f"Estimated time: ~{est_minutes:.0f} min ({est_minutes/60:.1f} h)")
print(f"Delay: {DELAY_BETWEEN_CALLS}s between calls")
print("API key configured OK")

In [ ]:
# ============================================================
# ALL 78 TAROT CARDS
# ============================================================

MAJOR_ARCANA = [
    {"id": "1",  "name": "O Mago",              "numeral": "I",    "type": "major"},
    {"id": "2",  "name": "A Papisa",             "numeral": "II",   "type": "major"},
    {"id": "3",  "name": "A Imperatriz",         "numeral": "III",  "type": "major"},
    {"id": "4",  "name": "O Imperador",          "numeral": "IV",   "type": "major"},
    {"id": "5",  "name": "O Papa",               "numeral": "V",    "type": "major"},
    {"id": "6",  "name": "O Amante",             "numeral": "VI",   "type": "major"},
    {"id": "7",  "name": "O Carro",              "numeral": "VII",  "type": "major"},
    {"id": "8",  "name": "A Justica",            "numeral": "VIII", "type": "major"},
    {"id": "9",  "name": "O Eremita",            "numeral": "IX",   "type": "major"},
    {"id": "10", "name": "A Roda da Fortuna",    "numeral": "X",    "type": "major"},
    {"id": "11", "name": "A Forca",              "numeral": "XI",   "type": "major"},
    {"id": "12", "name": "O Enforcado",          "numeral": "XII",  "type": "major"},
    {"id": "13", "name": "A Morte",              "numeral": "XIII", "type": "major"},
    {"id": "14", "name": "Temperanca",           "numeral": "XIV",  "type": "major"},
    {"id": "15", "name": "O Diabo",              "numeral": "XV",   "type": "major"},
    {"id": "16", "name": "A Casa Deus",          "numeral": "XVI",  "type": "major"},
    {"id": "17", "name": "A Estrela",            "numeral": "XVII", "type": "major"},
    {"id": "18", "name": "A Lua",                "numeral": "XVIII","type": "major"},
    {"id": "19", "name": "O Sol",                "numeral": "XIX",  "type": "major"},
    {"id": "20", "name": "O Julgamento",         "numeral": "XX",   "type": "major"},
    {"id": "21", "name": "O Mundo",              "numeral": "XXI",  "type": "major"},
    {"id": "22", "name": "O Louco",              "numeral": "0",    "type": "major"},
]

SUITS = [
    {"name": "Copas",   "element": "Agua",  "domain": "emocoes, amor, intuicao, relacionamentos"},
    {"name": "Paus",    "element": "Fogo",  "domain": "acao, criatividade, ambicao, energia"},
    {"name": "Espadas", "element": "Ar",    "domain": "mente, comunicacao, conflito, verdade"},
    {"name": "Ouros",   "element": "Terra", "domain": "materia, trabalho, saude, prosperidade"},
]

COURT_CARDS = ["Pajem", "Cavaleiro", "Rainha", "Rei"]
COURT_DESCRIPTIONS = {
    "Pajem":     "jovem aprendiz, mensageiro, potencial emergente",
    "Cavaleiro": "acao intensa, busca, movimento, idealismo",
    "Rainha":    "dominio interior, maturidade receptiva, nutricao",
    "Rei":       "dominio exterior, autoridade, maestria, lideranca",
}

PIP_MEANINGS = {
    "As":   "comeco puro, semente, potencial essencial",
    "2":    "dualidade, escolha, equilibrio, parceria",
    "3":    "expansao, crescimento, expressao criativa",
    "4":    "estabilidade, fundacao, pausa, estrutura",
    "5":    "crise, conflito, mudanca, desafio",
    "6":    "harmonia, reciprocidade, ajuste, comunicacao",
    "7":    "reflexao, avaliacao, desafio interior, fe",
    "8":    "movimento, poder, transformacao, dominio",
    "9":    "realizacao, culminacao, sabedoria, quase-completude",
    "10":   "completude, fim de ciclo, legado, transicao",
}

def build_minor_arcana():
    """Build all 56 Minor Arcana cards."""
    cards = []
    card_id = 23  # Starting after Major Arcana
    for suit in SUITS:
        pip_names = ["As", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
        for pip in pip_names:
            cards.append({
                "id": str(card_id),
                "name": f"{pip} de {suit['name']}",
                "numeral": pip,
                "type": "minor",
                "suit": suit["name"],
                "element": suit["element"],
                "domain": suit["domain"],
                "meaning": PIP_MEANINGS[pip],
            })
            card_id += 1
        for court in COURT_CARDS:
            cards.append({
                "id": str(card_id),
                "name": f"{court} de {suit['name']}",
                "numeral": court[0],
                "type": "minor",
                "suit": suit["name"],
                "element": suit["element"],
                "domain": suit["domain"],
                "meaning": COURT_DESCRIPTIONS[court],
            })
            card_id += 1
    return cards

MINOR_ARCANA = build_minor_arcana()
ALL_CARDS = MAJOR_ARCANA + MINOR_ARCANA
CARD_BY_ID = {c["id"]: c for c in ALL_CARDS}

print(f"Total cards: {len(ALL_CARDS)}")
print(f"Major Arcana: {len(MAJOR_ARCANA)}")
print(f"Minor Arcana: {len(MINOR_ARCANA)}")
print(f"Total directional pairs: {len(ALL_CARDS) * (len(ALL_CARDS) - 1)}")
print(f"\nSample minor cards:")
for c in MINOR_ARCANA[:3]:
    print(f"  {c['id']}: {c['name']} ({c['element']})")
print("  ...")
for c in MINOR_ARCANA[-3:]:
    print(f"  {c['id']}: {c['name']} ({c['element']})")

In [ ]:
# ============================================================
# PROMPT TEMPLATE & GENERATION HELPERS
# ============================================================

SYSTEM_PROMPT = """Voce e um especialista em Taro de Marselha com profundo conhecimento simbolico, \
arquetipico e esoterico. Voce escreve em portugues brasileiro formal e poetico.

Sua tarefa e descrever a DINAMICA DE INTERACAO entre pares de cartas do Taro.

Para cada par (Carta A -> Carta B), voce deve gerar 5 dinamicas:

1. **engendrar** - Como a energia da Carta A gera/cria/da origem a energia da Carta B.
2. **conflito** - Onde as energias das duas cartas colidem e se opoem.
3. **estagnar** - Como a combinacao pode levar a paralisia ou estagnacao.
4. **regressar** - A dinamica de retorno, diminuicao ou regressao entre elas.
5. **necessitar** - O que a Carta A precisa da Carta B para se completar.

REGRAS DE ESTILO:
- Cada dinamica: 2-3 frases (50-80 palavras), em portugues formal e poetico.
- Mencione o nome e o numeral/numero das cartas na primeira frase.
- Seja especifico sobre os simbolismos de cada carta.
- Para Arcanos Menores, considere o naipe (elemento) e o numero/corte.
- A interacao e DIRECIONAL: A->B e diferente de B->A.

FORMATO DE RESPOSTA: JSON puro, sem markdown code fences. Um array de objetos:
[
  {
    "source_id": "1",
    "target_id": "2",
    "dynamics": {
      "engendrar": "...",
      "conflito": "...",
      "estagnar": "...",
      "regressar": "...",
      "necessitar": "..."
    }
  }
]"""

def card_description(card):
    """Build a short description for the prompt."""
    if card["type"] == "major":
        return f"{card['name']} ({card['numeral']}) - Arcano Maior"
    else:
        return (f"{card['name']} - Arcano Menor, naipe de {card['suit']} "
                f"(elemento {card['element']}, dominio: {card['domain']}), "
                f"significado: {card['meaning']}")

def build_batch_prompt(pairs):
    """Build a prompt for a batch of card pairs."""
    lines = [f"Gere as dinamicas de interacao para os seguintes {len(pairs)} pares de cartas:\n"]
    for i, (source, target) in enumerate(pairs, 1):
        lines.append(f"--- Par {i} ---")
        lines.append(f'Carta A (source_id="{source["id"]}"): {card_description(source)}')
        lines.append(f'Carta B (target_id="{target["id"]}"): {card_description(target)}')
        lines.append("")
    lines.append("Responda APENAS com o JSON array, sem markdown code fences.")
    return "\n".join(lines)

def parse_response(text):
    """Parse JSON from the model response, handling markdown fences."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    return json.loads(text)

def generate_batch(pairs, retries=3):
    """Generate dynamics for a batch of pairs with retries."""
    prompt = build_batch_prompt(pairs)
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.8,
                    max_output_tokens=16384,
                ),
            )
            results = parse_response(response.text)
            # Validate structure
            for r in results:
                assert "source_id" in r and "target_id" in r and "dynamics" in r
                for key in ["engendrar", "conflito", "estagnar", "regressar", "necessitar"]:
                    assert key in r["dynamics"] and len(r["dynamics"][key]) > 20
            return results
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** (attempt + 2)  # 4s, 8s
                print(f"  Retry {attempt+1}/{retries} after: {str(e)[:80]}... waiting {wait}s")
                time.sleep(wait)
            else:
                raise

# Quick test with 1 pair
test_pairs = [(CARD_BY_ID["1"], CARD_BY_ID["23"])]
print("Testing API with 1 pair...")
test_result = generate_batch(test_pairs)
print(f"OK! Got {len(test_result)} result(s)")
print(f"Preview: {test_result[0]['dynamics']['engendrar'][:120]}...")

In [ ]:
# ============================================================
# PAIR GENERATION & CHECKPOINTING
# ============================================================

def generate_all_pairs():
    """Generate all directional pairs (A->B where A != B)."""
    pairs = []
    for source in ALL_CARDS:
        for target in ALL_CARDS:
            if source["id"] != target["id"]:
                pairs.append((source, target))
    return pairs

def load_checkpoint():
    """Load checkpoint data if it exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Checkpoint loaded: {data['completed_pairs']} pairs processed, "
              f"{len(data['results'])} results saved")
        return data
    return {"completed_pairs": 0, "results": {}, "errors": []}

def save_checkpoint(checkpoint):
    """Save checkpoint to disk."""
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False)

all_pairs = generate_all_pairs()
total_pairs = len(all_pairs)
total_batches = (total_pairs + PAIRS_PER_BATCH - 1) // PAIRS_PER_BATCH

print(f"Total pairs to generate: {total_pairs}")
print(f"Batch size: {PAIRS_PER_BATCH} pairs/call")
print(f"Total API calls needed: {total_batches}")
est_minutes = total_batches * DELAY_BETWEEN_CALLS / 60
print(f"Estimated time: ~{est_minutes:.0f} min ({est_minutes/60:.1f} h)")

In [ ]:
# ============================================================
# MAIN GENERATION LOOP (with checkpoint/resume)
# ============================================================
# Safe to re-run! It picks up exactly where it left off.

def run_generation():
    """Main generation loop with checkpointing."""
    checkpoint = load_checkpoint()
    results = checkpoint["results"]
    start_from = checkpoint["completed_pairs"]

    if start_from >= total_pairs:
        print("All pairs already generated! Skipping to output.")
        return results

    remaining = total_pairs - start_from
    remaining_batches = (remaining + PAIRS_PER_BATCH - 1) // PAIRS_PER_BATCH
    print(f"\nStarting from pair {start_from}/{total_pairs}")
    print(f"Remaining: {remaining} pairs in {remaining_batches} batches")
    print(f"{'='*60}\n")

    batch_num = start_from // PAIRS_PER_BATCH
    start_time = time.time()

    for i in range(start_from, total_pairs, PAIRS_PER_BATCH):
        batch_num += 1
        batch_pairs = all_pairs[i:i + PAIRS_PER_BATCH]

        # ETA calculation
        elapsed = time.time() - start_time
        done = i - start_from
        if done > 0:
            rate = done / elapsed
            eta_min = (total_pairs - i) / rate / 60
            eta_str = f" | ETA: {eta_min:.0f}min"
        else:
            eta_str = ""

        pair_desc = ", ".join(f"{s['name']}->{t['name']}" for s, t in batch_pairs[:3])
        print(f"[{batch_num}/{total_batches}] Pairs {i+1}-{min(i+PAIRS_PER_BATCH, total_pairs)}/{total_pairs}{eta_str}")
        print(f"  {pair_desc}...")

        try:
            batch_results = generate_batch(batch_pairs)
            for r in batch_results:
                key = f"{r['source_id']}_{r['target_id']}"
                results[key] = r
            print(f"  -> {len(batch_results)} pairs OK")
        except Exception as e:
            error_msg = f"Batch {batch_num}: {str(e)[:200]}"
            checkpoint["errors"].append(error_msg)
            print(f"  FAILED: {str(e)[:100]}")

        # Save checkpoint every batch
        checkpoint["completed_pairs"] = min(i + PAIRS_PER_BATCH, total_pairs)
        checkpoint["results"] = results
        save_checkpoint(checkpoint)

        # Rate limiting
        if i + PAIRS_PER_BATCH < total_pairs:
            time.sleep(DELAY_BETWEEN_CALLS)

    print(f"\n{'='*60}")
    print(f"Done! {len(results)} pairs generated.")
    if checkpoint["errors"]:
        print(f"Errors: {len(checkpoint['errors'])} (run retry cell below)")
    return results

results = run_generation()

In [ ]:
# ============================================================
# RETRY FAILED PAIRS (run if there were errors above)
# ============================================================

def retry_missing_pairs():
    """Find and retry any pairs missing from results."""
    checkpoint = load_checkpoint()
    results = checkpoint["results"]

    missing_pairs = []
    for source, target in all_pairs:
        key = f"{source['id']}_{target['id']}"
        if key not in results:
            missing_pairs.append((source, target))

    if not missing_pairs:
        print(f"No missing pairs! All {total_pairs} pairs are complete.")
        return results

    print(f"Found {len(missing_pairs)} missing pairs. Retrying...\n")

    for i in range(0, len(missing_pairs), PAIRS_PER_BATCH):
        batch = missing_pairs[i:i + PAIRS_PER_BATCH]
        batch_num = i // PAIRS_PER_BATCH + 1
        total = (len(missing_pairs) + PAIRS_PER_BATCH - 1) // PAIRS_PER_BATCH
        print(f"Retry batch {batch_num}/{total}...")
        try:
            batch_results = generate_batch(batch)
            for r in batch_results:
                key = f"{r['source_id']}_{r['target_id']}"
                results[key] = r
            print(f"  -> Recovered {len(batch_results)} pairs")
        except Exception as e:
            print(f"  Still failing: {str(e)[:100]}")

        checkpoint["results"] = results
        save_checkpoint(checkpoint)
        time.sleep(DELAY_BETWEEN_CALLS)

    still_missing = sum(1 for s, t in all_pairs if f"{s['id']}_{t['id']}" not in results)
    print(f"\nDone. Still missing: {still_missing} pairs")
    return results

results = retry_missing_pairs()

In [ ]:
# ============================================================
# BUILD FINAL OUTPUT JSON
# ============================================================

def build_output_json(results):
    """Build the final JSON in the same format as the original tarot_data.json."""
    output = []

    for card in ALL_CARDS:
        card_entry = {
            "id": card["id"],
            "name": card["name"],
            "interactions": []
        }

        if card["type"] == "minor":
            card_entry["suit"] = card["suit"]
            card_entry["element"] = card["element"]

        for target in ALL_CARDS:
            if target["id"] == card["id"]:
                continue

            key = f"{card['id']}_{target['id']}"
            if key in results:
                interaction = {
                    "target_id": target["id"],
                    "target_name": target["name"],
                    "dynamics": results[key]["dynamics"]
                }
                card_entry["interactions"].append(interaction)

        output.append(card_entry)

    return output

output_data = build_output_json(results)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
total_interactions = sum(len(c["interactions"]) for c in output_data)

print(f"Output saved to: {OUTPUT_FILE}")
print(f"File size: {file_size:.1f} MB")
print(f"Cards: {len(output_data)}")
print(f"Total interactions: {total_interactions}")
print(f"Expected: {78 * 77}")
print(f"Coverage: {total_interactions / (78 * 77) * 100:.1f}%")

if output_data:
    sample = output_data[0]
    print(f"\nSample - {sample['name']}:")
    print(f"  Interactions: {len(sample['interactions'])}")
    if sample['interactions']:
        first = sample['interactions'][0]
        print(f"  First target: {first['target_name']}")
        print(f"  Engendrar: {first['dynamics']['engendrar'][:120]}...")

In [ ]:
# ============================================================
# DONE! Your files are saved to Google Drive
# ============================================================

print("=" * 50)
print("ALL FILES SAVED TO GOOGLE DRIVE")
print("=" * 50)
print(f"\nFolder: My Drive/tarot-generator/")
print(f"  - tarot_data_full.json  (the final result)")
print(f"  - checkpoint.json       (resume data)")
print(f"\nFile size: {os.path.getsize(OUTPUT_FILE) / (1024*1024):.1f} MB")
print(f"\nTo use it:")
print(f"  1. Go to Google Drive -> tarot-generator/")
print(f"  2. Download tarot_data_full.json")
print(f"  3. Rename to tarot_data.json and replace the original in your repo")
print(f"\nOr copy directly from Drive to your local clone:")
print(f'  cp "{OUTPUT_FILE}" /path/to/rampa/tarot-reader/tarot_data.json')

In [ ]:
# ============================================================
# DOWNLOAD
# ============================================================

from google.colab import files

print("Downloading tarot_data_full.json...")
files.download(OUTPUT_FILE)
print("\nDone! Rename this to tarot_data.json and replace the original.")
print("Then update app.js to include all 78 cards in the UI.")